# Weeks 4–5 · Solutions

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
def load_clean_students():
    """Shared loader used by several exercises."""
    raw = pd.read_csv(DATA / "students.csv")
    return (raw.assign(major=lambda d: d["major"].str.strip(),
                       city=lambda d: d["city"].str.title(),
                       gpa=lambda d: pd.to_numeric(d["gpa"], errors="coerce"))
               .drop_duplicates().reset_index(drop=True))
students = load_clean_students()
enr = pd.read_csv(DATA / "enrollments.csv")
sales = pd.read_csv(DATA / "sales_2025.csv", parse_dates=["date"])
print(students.shape, enr.shape, sales.shape)

### Exercise 1 · Selection

In [ ]:
def top_irbid(df):
    m = df["year"].isin([3, 4]) & (df["city"] == "Irbid") & (df["gpa"] >= 3.5)
    return df.loc[m, ["name", "gpa"]].sort_values(["gpa", "name"], ascending=[False, True])

In [ ]:
r = top_irbid(students)
assert list(r.columns) == ["name", "gpa"]
assert (r["gpa"] >= 3.5).all() and r["gpa"].is_monotonic_decreasing
assert set(students.loc[r.index, "city"]) == {"Irbid"} and set(students.loc[r.index, "year"]) <= {3, 4}
print("✓ exercise 1 -", len(r), "rows")

### Exercise 2 · Cleaning audit

In [ ]:
def audit(raw):
    gpa_num = pd.to_numeric(raw["gpa"], errors="coerce")
    blank = raw["gpa"].isna() | (raw["gpa"].astype(str).str.strip() == "")
    return {
        "duplicates": int(raw.duplicated().sum()),
        "bad_gpa": int((gpa_num.isna() & ~blank).sum()),
        "blank_gpa": int(blank.sum()),
        "untrimmed_major": int((raw["major"] != raw["major"].str.strip()).sum()),
        "upper_city": int(raw["city"].str.isupper().sum()),
    }

In [ ]:
raw = pd.read_csv(DATA / "students.csv")
a = audit(raw)
assert set(a) == {"duplicates", "bad_gpa", "blank_gpa", "untrimmed_major", "upper_city"}
assert a["duplicates"] == 5 and a["bad_gpa"] > 0 and a["blank_gpa"] > 0 and a["untrimmed_major"] > 0 and a["upper_city"] > 0
print("✓ exercise 2", a)

### Exercise 3 · Missing values, group-wise

In [ ]:
def fill_gpa_by_major(df):
    out = df.copy()
    out["gpa"] = out["gpa"].fillna(out.groupby("major")["gpa"].transform("median"))
    return out

In [ ]:
f = fill_gpa_by_major(students)
assert f["gpa"].isna().sum() == 0 and students["gpa"].isna().sum() > 0
m = students["gpa"].isna()
for major in students.loc[m, "major"].unique():
    med = students.loc[students["major"] == major, "gpa"].median()
    assert (f.loc[m & (students["major"] == major), "gpa"] == med).all()
print("✓ exercise 3")

### Exercise 4 · Derived columns

In [ ]:
def derive(df):
    return df.assign(
        progress=(df["credits"] / 132 * 100).round(1),
        band=pd.cut(df["gpa"], bins=[0, 2, 3, 4.0001], labels=["C", "B", "A"], right=False),
    )

In [ ]:
d = derive(students)
assert "progress" in d and "band" in d
assert abs(d.loc[0, "progress"] - round(students.loc[0, "credits"] / 132 * 100, 1)) < 1e-9
assert d.loc[d["gpa"] == 4.0, "band"].astype(str).eq("A").all()
assert d.loc[d["gpa"] < 2.0, "band"].astype(str).eq("C").all()
print("✓ exercise 4", d["band"].value_counts().to_dict())

### Exercise 5 · GroupBy summary

In [ ]:
def city_report(df):
    return (df.groupby("city")
              .agg(students=("student_id", "size"),
                   mean_gpa=("gpa", "mean"),
                   top_major=("major", lambda s: s.mode()[0]))
              .round({"mean_gpa": 2})
              .sort_values("students", ascending=False))

In [ ]:
r = city_report(students)
assert list(r.columns) == ["students", "mean_gpa", "top_major"]
assert r["students"].is_monotonic_decreasing and r["students"].sum() == len(students)
assert r.loc["Amman", "top_major"] == students[students["city"] == "Amman"]["major"].mode()[0]
print("✓ exercise 5"); display(r)

### Exercise 6 · Merge

In [ ]:
def grades_with_names(students, enr):
    return (enr.merge(students[["student_id", "name", "major"]], on="student_id", how="inner")
               .dropna(subset=["grade"])
               [["student_id", "name", "major", "course_code", "grade"]]
               .reset_index(drop=True))

In [ ]:
g = grades_with_names(students, enr)
assert list(g.columns) == ["student_id", "name", "major", "course_code", "grade"]
assert g["grade"].notna().all() and g["name"].notna().all()
assert len(g) == enr["grade"].notna().sum()
print("✓ exercise 6 -", len(g), "rows")

### Exercise 7 · Pivot

In [ ]:
def grade_pivot(g):
    p = g.pivot_table(index="major", columns="course_code", values="grade", aggfunc="mean")
    p["overall"] = g.groupby("major")["grade"].mean()
    return p.round(1)

In [ ]:
p = grade_pivot(grades_with_names(students, enr))
assert "overall" in p.columns and p.index.name == "major"
ds = grades_with_names(students, enr).query("major == 'Data Science'")["grade"].mean().round(1)
assert abs(p.loc["Data Science", "overall"] - ds) < 1e-9
print("✓ exercise 7"); display(p)

### Exercise 8 · Time series

In [ ]:
def monthly_product_revenue(sales):
    s = sales.assign(revenue=sales["units"] * sales["unit_price"])
    return (s.pivot_table(index="date", columns="product", values="revenue", aggfunc="sum")
             .resample("ME").sum().round(0))

In [ ]:
m = monthly_product_revenue(sales)
assert m.shape == (12, 5) and set(m.columns) == set(sales["product"].unique())
assert m.index[0].month == 1 and m.index[-1].month == 12
tot = (sales["units"] * sales["unit_price"]).sum()
assert abs(m.values.sum() - tot) < 5
print("✓ exercise 8"); display(m.head(3))

### Exercise 9 · Rolling + ranking

In [ ]:
def best_week(sales):
    daily = sales.assign(rev=sales["units"] * sales["unit_price"]).groupby("date")["rev"].sum()
    return daily.rolling(7).sum().idxmax()

In [ ]:
b = best_week(sales)
assert isinstance(b, pd.Timestamp)
daily = sales.assign(rev=sales["units"] * sales["unit_price"]).groupby("date")["rev"].sum()
assert b == daily.rolling(7).sum().idxmax()
print("✓ exercise 9", b.date())

### Exercise 10 · Mini EDA function

In [ ]:
def eda(df):
    missing = df.isna().sum()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    return {
        "rows": len(df),
        "cols": df.shape[1],
        "missing": {c: int(v) for c, v in missing.items() if v > 0},
        "numeric": df.select_dtypes(include="number").columns.tolist(),
        "top_categories": {c: df[c].mode()[0] for c in cat_cols if df[c].nunique() <= 10},
    }

In [ ]:
e = eda(students)
assert e["rows"] == len(students) and e["cols"] == students.shape[1]
assert e["missing"] == {"gpa": int(students["gpa"].isna().sum())}
assert set(e["numeric"]) == {"student_id", "year", "gpa", "credits"}
assert set(e["top_categories"]) == {"major", "city"}
print("✓ exercise 10", e)